In [ ]:
input_data = None
input_doc = None
output_data = None
util = None
display_util = None

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline

# from IPython.display import Markdown

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rename,
    display_data_doc,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    drop_duplicate_columns,
)

# empfaenger

This data file provides information on the recipients of an organ transplantation. Data is provided by the {term}`ET` foundation and the {term}`IQTIG` institute in the file `element_empfaenger.csv`. 

For measurements like the body weight for example there are no update dates available in this file. 

We perform the common steps outlined in [](../02_preprocessing/index.md). The integration of the seperated institute data is not necessary, because they are already correctly joined. Furthermore unit conversions are not necessary beside small translations of existing values.


## Unprocessed input data

In [ ]:
data = pd.read_csv(input_data, sep=";", low_memory=False)
display_data_doc(data=data, official_doc=pd.read_csv(input_doc))

## Technical Steps

For this file the general plan for technical preprocessing was followed (see [](general:ts)).

### Removal of Non-Informative Columns

First empty and duplicated columns were removed if there are any (see [](general:ecf)). Furthermore we remove the columns `EBasisPLZET`, `EBasisZentrumRegistrierungET` and `EBasisZentrumKontaktierungET` as they only contain encoded identifying information and the column `EBasisGeburtsdatumIQTIG`, as it only contains invalid values.

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

assert data["EBasisGeburtsdatumIQTIG"].nunique() == 1, "IQTIG Birthday now valid!?"

data.drop(
    [
        "EBasisPLZET",
        "EBasisZentrumRegistrierungET",
        "EBasisZentrumKontaktierungET",
        "EBasisGeburtsdatumIQTIG",
    ],
    axis=1,
    inplace=True,
)

### Renaming of Columns

New names are used for the columns (see [](general:cr)).

In [ ]:
renaming = {
    "EBasisAnzahlKinderET": "children",
    "EBasisAnzahlSchwangerschaftenET": "pregnancies",
    "EBasisBlutgrET": "bloodgroup_et",
    "EBasisBlutgrIQTIG": "bloodgroup_iqtig",
    "EBasisBluttransfusionNachRegistrierungET": "bloodtransfusion_after_reg",
    "EBasisBluttransfusionVorRegistrierungET": "bloodtransfusion_before_reg",
    "EBasisGeburtsdatumET": "birthdate",
    "EBasisGeschlechtIQTIG": "sex_iqtig",
    "EBasisGeschlechtET": "sex_et",
    "EBasisGewichtWertET": "weight_kg_et",
    "EBasisGewichtWertIQTIG": "weight_kg_iqtig",
    "EBasisGroesseWertET": "height_cm_et",
    "EBasisGroesseWertIQTIG": "height_cm_iqtig",
    "EBasisGewichtEinheitET": "weight_unit_et",
    "EBasisGewichtEinheitIQTIG": "weight_unit_iqtig",
    "EBasisGroesseEinheitET": "height_unit_et",
    "EBasisGroesseEinheitIQTIG": "height_unit_iqtig",
    "EBasisTodesdatumET": "death_date",
    "EBasisLandET": "origin_country",
    "EBasisNationalitaetET": "nationality",
    "EBasisRhesusfaktorET": "rhesus_et",
    "EBasisRhesusfaktorIQTIG": "rhesus_iqtig",
    "EBasisTodesursacheET": "death_cause",
    "EIdEmpfaengerNrETET": "recipient_et_id_et",
    "EIdEmpfaengerNrETIQTIG": "recipient_et_iqtig",
}

# renaming = dict(zip(data.columns.to_list(), "RENAMEME_" + data.columns))

# template = ",\n".join([f'    "{col}":"RENAMEME_{col}"' for col in data.columns])
# template = "renaming = {\n" + template + ",\n}"
# print(template)

data = rename(data, renaming)
data.to_parquet(output_data)